In [ ]:
%%capture
import os
from pathlib import Path
import pandas as pd

from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
reports_folder = Path(os.environ["INTECOMM_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)

In [ ]:
from django_pandas.io import read_frame
from intecomm_analytics.dataframes import get_patientlog_df
from intecomm_rando.models import RandomizationList
from intecomm_analytics.dataframes import get_df_main_1858

In [ ]:
df_main = get_df_main_1858(None)

In [ ]:
df_rando = read_frame(RandomizationList.objects.all())

In [ ]:
df_rando[(df_rando.allocation=="2") & (df_rando.group_identifier.notna())]

In [ ]:
df_rando[df_rando.group_identifier.notna()][["group_identifier", "assignment", "allocation"]]

In [ ]:
df_rando[df_rando.group_identifier.notna()][["group_identifier", "assignment", "allocation"]].assignment.value_counts()

In [ ]:
df = pd.merge(df_main[["subject_identifier", "group_identifier", "assignment", "allocation"]], df_rando[["group_identifier", "assignment", "allocation"]], on="group_identifier", suffixes=("_1858", "_original"))

In [ ]:
df

In [ ]:
df = pd.read_csv('/Users/erikvw/Documents/ucl/protocols/intecomm/export/randomization_list_expanded.csv')
df = df.groupby(by=["assignment", "site_name"]).size().reset_index(name="total")
df_pivot = df.pivot_table(index='site_name', columns='assignment', values='total', fill_value=0).reset_index()
df_pivot.columns.name = None
df_pivot['total'] = df_pivot['a'] + df_pivot['b']
totals = df_pivot.sum(numeric_only=True)
totals['site_name'] = 'Total'
df_pivot = pd.concat([df_pivot, totals.to_frame().T], ignore_index=True)

In [ ]:
display(df_pivot)

In [ ]:
df = get_df_main_1858(None)
df = df.groupby(by=["assignment", "site"]).size().reset_index(name="total")
df_pivot = df.pivot_table(index='site', columns='assignment', values='total', fill_value=0).reset_index()
df_pivot.columns.name = None
df_pivot['total'] = df_pivot['a'] + df_pivot['b']
totals = df_pivot.sum(numeric_only=True)
totals['site'] = 'Total'
df_pivot = pd.concat([df_pivot, totals.to_frame().T], ignore_index=True)
df_pivot

In [ ]:
display(df_pivot)


In [ ]:
df_main[["assignment", "hiv_only", "ncd"]].groupby(by=["assignment", "ncd"]).size()

In [ ]:


df_plog = get_patientlog_df()

In [ ]:
df_plog[(df_plog.consent_datetime.notna()) & (df_plog.group_identifier.notna())]

In [ ]:
from intecomm_analytics.dataframes.df_main_1858.get_df_main_1858 import merge_in_visit

# start with patient log
# using patient_log is one way to link group_identifier and subject_identifier
df_main = get_patientlog_df()

# exclude those in patient_log that were not added to a group
df_main = df_main[(df_main.group_identifier.notna())]

# exclude those added to a group but never consented
df_main = df_main[(df_main.consent_datetime.notna())]

assert len(df_main) == 1864

# rename conditions reported at screening to distinguish from those
# confirmed later at baseline
df_main.rename(columns={"hiv": "hiv_scr", "htn": "htn_scr", "dm": "dm_scr"}, inplace=True)

# merge with df_visit
# this merge leaves us with only the subjects who presented for the
# rando / baseline visit
df_main = merge_in_visit(df_main)

assert len(df_main) == 1858


In [ ]:
from intecomm_analytics.dataframes.df_main_1858.get_df_main_1858_pre import merge_in_rando

df_main = merge_in_rando(df_main)


In [ ]:
df_main.assignment.value_counts()

In [ ]:
assert len(df_main[df_main.assignment == "a"]) == 932
assert len(df_main[df_main.assignment == "b"]) == 926


In [ ]:
assert len(df_main[df_main.allocation == "1"]) == 932
assert len(df_main[df_main.allocation == "2"]) == 926


In [ ]:
df_main[(df_main.hiv_only==1)]

In [ ]:
df_hiv = df_main[(df_main.hiv_only==1) & (df_main.hiv_timedelta_dx.dt.days>=180)].copy()


In [ ]:
df_main[(df_main.hiv_only==1)].hiv_timedelta_dx.dt.days.describe()

In [ ]:
df_dm = df_main[(df_main.dm_only==1) & (df_main.dm_timedelta_dx.dt.days>=180)].copy()


In [ ]:
df_main[(df_main.dm_only==1)].dm_timedelta_dx.dt.days.describe()


In [ ]:
df_htn = df_main[(df_main.htn_only==1) & (df_main.htn_timedelta_dx.dt.days>=180)].copy()


In [ ]:
df_main[(df_main.htn_only==1)].htn_timedelta_dx.dt.days.describe()


In [ ]:
df_and = df_main[(df_main.htn_and_dm==1) & (df_main.dm_timedelta_dx.dt.days>=180) & (df_main.htn_timedelta_dx.dt.days>=180)].copy()

In [ ]:
df = pd.concat([df_hiv, df_dm, df_htn, df_and])

In [ ]:
df

In [ ]:
df.groupby(by=["hiv_only", "ncd"]).size()

In [ ]:
df.groupby(by=["assignment", "hiv_only"]).size()

In [ ]:
df_main[[col for col in df_main.columns if col.startswith("vl")]]